https://www.youtube.com/watch?v=oXlwWbU8l2o
https://debuggercafe.com/transfer-learning-using-efficientnet-pytorch/

In [2]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

#run cell, click into kaggle folder, then run cell again
import kagglehub
alexattia_the_simpsons_characters_dataset_path = kagglehub.dataset_download('alexattia/the-simpsons-characters-dataset')

print('Data source import complete.')
print(alexattia_the_simpsons_characters_dataset_path)


Using Colab cache for faster access to the 'the-simpsons-characters-dataset' dataset.
Data source import complete.
/kaggle/input/the-simpsons-characters-dataset


##Preparing dataset for training w/ EfficientNet

In [3]:
IMG_SIZE=(224, 224)
VALID_SIZE=0.2 #20% of training set should go to validation set
BATCH_SIZE=128

In [4]:
import torchvision.transforms as torchvision_T

def normalize(IMG_SIZE, pretrained=True):
  if pretrained:
    return torchvision_T.Normalize(mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225])
  else:
    return torchvision_T.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])

def train_transforms(IMG_SIZE, pretrained=True):
  transforms=torchvision_T.Compose([torchvision_T.Resize(IMG_SIZE), torchvision_T.RandomGrayscale(p=0.4), torchvision_T.RandomHorizontalFlip(p=0.5),
                                    torchvision_T.RandomAdjustSharpness(2, 0.5), torchvision_T.GaussianBlur((5, 9), (0.1, 5)), torchvision_T.ToTensor(), normalize(IMG_SIZE, pretrained)])
  return transforms

def valid_transforms(IMG_SIZE, pretrained=True):
  transforms=torchvision_T.Compose([torchvision_T.Resize(IMG_SIZE), torchvision_T.ToTensor(), normalize(IMG_SIZE, pretrained)])
  return transforms

In [5]:
from torchvision import datasets
import torch
from torch.utils.data import Dataset, DataLoader, Subset

def get_datasets(pretrained=True):
  dataset=datasets.ImageFolder(root_dir+"/simpsons_dataset"+"/simpsons_dataset", transform=train_transforms(IMG_SIZE))

  sz=len(dataset)
  valid_size=int(VALID_SIZE*sz)
  indices=torch.randperm(len(dataset)).tolist()
  dataset_train=Subset(dataset, indices[:-valid_size])
  dataset_valid=Subset(dataset, indices[-valid_size:])

  return dataset_train, dataset_valid, dataset.classes

def get_loaders(pretrained=True):
  train, valid, classes=get_datasets()
  print("# classes: ", len(classes))
  print(classes)
  train_loader=DataLoader(train, batch_size=BATCH_SIZE, shuffle=True)
  valid_loader=DataLoader(valid, batch_size=BATCH_SIZE, shuffle=False)
  return train_loader, valid_loader

##Building model (EfficientNetb0)

In [6]:
from torchvision.models import efficientnet_b0

def build_model(num_classes=42, model_name='efficientnetb0', pretrained=True):
  if pretrained:
    if model_name=='efficientnetb0':
      model=efficientnet_b0(weights='DEFAULT')

  #for finetuning, set requires_grad to True in order to train intermediate layers
  for params in model.parameters():
    params.requires_grad=True

  #modifying classifier layer to fit num_classes
  model.classifier[1]=torch.nn.Linear(in_features=1280, out_features=num_classes)
  return model

##Training and validation steps

In [7]:
from tqdm import tqdm

def training_step(model, loader, epoch, optimizer, criterion):
  model.train()
  training_loss=0.0
  training_num_correct=0
  counter=0
  for i, data in tqdm(enumerate(loader), desc=f'train :: epoch {str(epoch)}', total=len(loader)):
    counter+=1

    #passing training data through the model
    img, labels=data
    img=img.to(device)
    labels=labels.to(device)
    preds=model(img)

    #calculating loss
    curr_loss=criterion(preds, labels)
    training_loss+=curr_loss.item()

    #calculating accuracy
    _, preds=torch.max(preds.data, 1)
    training_num_correct+=(preds==labels).sum().item()

    #backpropagation
    curr_loss.backward()

    optimizer.step()

  #epoch statistics
  epoch_loss=training_loss/counter #mean loss
  epoch_acc=100.*(training_num_correct/len(loader.dataset))

  return epoch_loss, epoch_acc

In [8]:
def valid_step(model, loader, epoch, criterion):
  model.eval()
  valid_loss=0.0
  valid_correct=0
  counter=0
  with torch.no_grad():
    for i, data in tqdm(enumerate(loader), desc=f"valid :: epoch {str(epoch)}", total=len(loader)):
      counter+=1

      #run data through the model
      img, labels=data
      img=img.to(device)
      labels=labels.to(device)
      preds=model(img).detach()

      #calculate loss
      loss=criterion(preds, labels)
      valid_loss+=loss.item()

      #calculating accuracy
      _, preds=torch.max(preds.data, 1)
      valid_correct+=(preds==labels).sum().item()

  #epoch statistics
  epoch_loss=valid_loss/counter #mean loss
  epoch_acc=100.*(valid_correct/len(loader.dataset))

  return epoch_loss, epoch_acc


##Training the model

In [9]:
#Hyperparameters

EPOCHS=25
LEARNING_RATE=0.0001
device='cuda' if torch.cuda.is_available() else 'cpu'
MODEL_PATH='best_model.pth'
BATCH_SIZE=128

In [ ]:
root_dir="/kaggle/input/the-simpsons-characters-dataset"
train_loader, valid_loader=get_loaders()

# classes:  42
['abraham_grampa_simpson', 'agnes_skinner', 'apu_nahasapeemapetilon', 'barney_gumble', 'bart_simpson', 'carl_carlson', 'charles_montgomery_burns', 'chief_wiggum', 'cletus_spuckler', 'comic_book_guy', 'disco_stu', 'edna_krabappel', 'fat_tony', 'gil', 'groundskeeper_willie', 'homer_simpson', 'kent_brockman', 'krusty_the_clown', 'lenny_leonard', 'lionel_hutz', 'lisa_simpson', 'maggie_simpson', 'marge_simpson', 'martin_prince', 'mayor_quimby', 'milhouse_van_houten', 'miss_hoover', 'moe_szyslak', 'ned_flanders', 'nelson_muntz', 'otto_mann', 'patty_bouvier', 'principal_skinner', 'professor_john_frink', 'rainier_wolfcastle', 'ralph_wiggum', 'selma_bouvier', 'sideshow_bob', 'sideshow_mel', 'snake_jailbird', 'troy_mcclure', 'waylon_smithers']


In [12]:
model=build_model()
criterion=torch.nn.CrossEntropyLoss() #can assign weights to each of the classes
optimizer=torch.optim.Adam(model.parameters(), lr=0.0001)
model.to(device)

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivat

In [13]:
train_loss_log, train_acc_log=[], []
valid_loss_log, valid_acc_log=[], []
best_acc=0.0
torch.cuda.empty_cache()

for epoch in range(1, EPOCHS+1):
  tloss, tacc=training_step(model, train_loader, epoch, optimizer, criterion)
  vloss, vacc=valid_step(model, valid_loader, epoch, criterion)

  train_loss_log.append(tloss)
  train_acc_log.append(tacc)
  valid_loss_log.append(vloss)
  valid_acc_log.append(vacc)

  if vacc>=best_acc:
    torch.save(model.state_dict(), MODEL_PATH)

  print('-'*50)

valid :: epoch 1: 100%|██████████| 33/33 [01:16<00:00,  2.32s/it]


--------------------------------------------------


valid :: epoch 2: 100%|██████████| 33/33 [00:52<00:00,  1.60s/it]


--------------------------------------------------


valid :: epoch 3: 100%|██████████| 33/33 [00:53<00:00,  1.62s/it]


...

--------------------------------------------------


valid :: epoch 24: 100%|██████████| 33/33 [00:53<00:00,  1.62s/it]


--------------------------------------------------


valid :: epoch 25: 100%|██████████| 33/33 [00:54<00:00,  1.64s/it]

--------------------------------------------------
